In [1]:
import glob
import math
import numpy as np
import concurrent.futures

import ROOT
from utils.hist_utils import get_highest_channels, fit_and_scale_mc
from utils.physics_utils import add_charged_kaon_mass

ROOT.EnableImplicitMT(100)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/Include/Codes/")

ROOT.gInterpreter.Declare('#include "const.h"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

In [ ]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS2_THREE_PI0_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS3_THREE_PI0_NoSmearing/*.root"
)

data_files = []

for i in range(1, 211, 1):
    data_files += glob.glob(
        f"/workspace/root_files/hal/DATA_THREE_PI0_NoSmearing/dk0*_{i}.root"
    )

def check_single_file(f):
    """Sprawdza pojedynczy plik na osobnym wątku."""
    try:
        # kError ignoruje niepotrzebne ostrzeżenia w konsoli
        ROOT.gErrorIgnoreLevel = ROOT.kError
        tf = ROOT.TFile.Open(f, "READ")
        if tf and not tf.IsZombie():
            tree = tf.Get("h1")
            if tree and not tree.IsZombie():
                tf.Close()
                return f
        if tf:
            tf.Close()
    except Exception:
        pass
    return None


def get_valid_files_fast(file_list):
    """Sprawdza całą listę plików równolegle na wielu rdzeniach CPU."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=64) as executor:
        results = list(executor.map(check_single_file, file_list))
    # Zwraca tylko poprawne pliki
    valid = [f for f in results if f is not None]
    print(
        f"Przetestowano {len(file_list)} plików. Poprawnych: {len(valid)}, Odrzuconych: {len(file_list) - len(valid)}"
    )
    return valid


# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

# Scaling factor
lumi_data = 817216.9875
lumi_mc = 745800.1127
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

me = 0.511  # MeV
mmu = 105.659  # MeV
mpich = 139.570  # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data.Define("Kchrec_5", "Kchrec[5]")
    .Define("Knerec_5", "Knerec[5] - 497.611")
    .Define("KnerecSix_5", "KnerecSix[5] - 497.611")
    .Define("KnetriKinFit_5", "KnetriKinFit[5] - 497.611")
    .Define("MassPi01", "pi01[5]")
    .Define("MassPi02", "pi02[5]")
    .Define("rho_ch", "hypot(Kchrec[6] - ip[0], Kchrec[7] - ip[1])")
    .Define("rho_ne", "hypot(Knerec[6] - Bx, Knerec[7] - By)")
    .Define("z_ch", "Kchrec[8] - ip[2]")
    .Define("z_ne", "abs(Knerec[8] - Bz)")
    .Define(
        "rho_fid_omega", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])"
    )
    .Define("z_fid_omega", "Kchrec[8] - Knerec[8]")
    .Filter("mctruth != -1")
)

# Obliczenie mas surowych dla danych
df_data = add_charged_kaon_mass(df_data, "m_pie", mpich, me)
df_data = add_charged_kaon_mass(df_data, "m_epi", me, mpich)
df_data = add_charged_kaon_mass(df_data, "m_pimu", mpich, mmu)
df_data = add_charged_kaon_mass(df_data, "m_mupi", mmu, mpich)

# Definicje hipotez masowych na podstawie Curv1 dla danych
df_data = (
    df_data.Define("m_e_plus_hyp", "Curv1 > 0 ? m_pie : m_epi")
    .Define("m_e_minus_hyp", "Curv1 > 0 ? m_epi : m_pie")
    .Define("m_mu_plus_hyp", "Curv1 > 0 ? m_pimu : m_mupi")
    .Define("m_mu_minus_hyp", "Curv1 > 0 ? m_mupi : m_pimu")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df.Define("Kchrec_5", "Kchrec[5]")
    .Define("Knerec_5", "Knerec[5] - 497.611")
    .Define("KnerecSix_5", "KnerecSix[5] - 497.611")
    .Define("KnetriKinFit_5", "KnetriKinFit[5] - 497.611")
    .Define("MassPi01", "pi01[5]")
    .Define("MassPi02", "pi02[5]")
    .Define("rho_ch", "hypot(Kchrec[6] - ip[0], Kchrec[7] - ip[1])")
    .Define("rho_ne", "hypot(Knerec[6] - Bx, Knerec[7] - By)")
    .Define("z_ch", "Kchrec[8] - ip[2]")
    .Define("z_ne", "abs(Knerec[8] - Bz)")
    .Define(
        "rho_fid_omega", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])"
    )
    .Define("z_fid_omega", "Kchrec[8] - Knerec[8]")
    .Filter("mctruth != -1 && mctruth != 0 ")
)

# Obliczenie mas surowych dla MC
df = add_charged_kaon_mass(df, "m_pie", mpich, me)
df = add_charged_kaon_mass(df, "m_epi", me, mpich)
df = add_charged_kaon_mass(df, "m_pimu", mpich, mmu)
df = add_charged_kaon_mass(df, "m_mupi", mmu, mpich)

# Definicje hipotez masowych na podstawie Curv1 dla MC
df = (
    df.Define("m_e_plus_hyp", "Curv1 > 0 ? m_pie : m_epi")
    .Define("m_e_minus_hyp", "Curv1 > 0 ? m_epi : m_pie")
    .Define("m_mu_plus_hyp", "Curv1 > 0 ? m_pimu : m_mupi")
    .Define("m_mu_minus_hyp", "Curv1 > 0 ? m_mupi : m_pimu")
)

# Filtrowanie dla próbki semileptonicznej (dziedziczy wszystkie nowe kolumny m_*_hyp)
df_Semileptonic = df.Filter(
    "mcflag == 1 && (mctruth == 5 || mctruth == 0 || mctruth == -1)"
)

## Optimization of the cut for the control sample tag $K\to\pi^+\pi^-$

In [ ]:
df_CS_tag = {}

df_CS_tag["00"] = df.Filter("has00 && mctruth != 0 && mctruth != -1")
df_CS_tag["non00"] = df.Filter("!has00 && mctruth != 0 && mctruth != -1")
df_CS_tag["data"] = df_data.Filter("mctruth != -1")

total_counts = {}

total_counts["00"] = df_CS_tag["00"].Count()
total_counts["non00"] = df_CS_tag["non00"].Count()
total_counts["data"] = df_CS_tag["data"].Count()

ROOT.RDF.RunGraphs(list(total_counts.values()))

total_counts = {key: int(val.GetValue() * scaling_factor) if key != 'data' else int(val.GetValue()) for key, val in total_counts.items()}

### Create the histograms

In [ ]:
cut_cond = "true"
optimized_variable = "Chi2TriKinFit"
var_title = "#chi^{2}_{tri} [-]"

In [ ]:
basic_hist = {}
modes = {"full": [201, 0, 200], "zoom": [201, 0, 80]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["00", "non00", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: 00, Index 1: non00, Index 2: data
    
    # Skalowanie histogramów MC (00 i non00)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (00 + non00) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [ ]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 2000, 1000)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow#pi^{0}#pi^{0}", 
    "Events without primary K#rightarrow#pi^{0}#pi^{0}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []
lines = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    # line_left = ROOT.TLine(497.611 - 2.0, 0.0, 497.611 - 2.0, hs.GetMaximum() * 0.5)
    # line_right = ROOT.TLine(4.0, 0.0, 4.0, hs.GetMaximum() * 0.5)

    # line_right.SetLineStyle(ROOT.kDashed)
    
    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    # line_right.Draw("SAME")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)
    # lines.append(line_right)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

### Optimization loop

In [ ]:
tag_condition = "rho_ne < 4.0 && abs(z_ne) < 2.0 && abs(Knerec_5) < 130.0"

df_CS_tag["00"] = df_CS_tag["00"].Filter(tag_condition)
df_CS_tag["non00"] = df_CS_tag["non00"].Filter(tag_condition)
df_CS_tag["data"] = df_CS_tag["data"].Filter(tag_condition)

total_counts_fv = {}

total_counts_fv["00"] = df_CS_tag["00"].Count()
total_counts_fv["non00"] = df_CS_tag["non00"].Count()
total_counts_fv["data"] = df_CS_tag["data"].Count()

ROOT.RDF.RunGraphs(list(total_counts_fv.values()))

total_counts_fv = {key: int(val.GetValue() * scaling_factor) if key != 'data' else int(val.GetValue()) for key, val in total_counts_fv.items()}

In [ ]:
cut_values = [rms * float(val) for val in np.arange(0, 5.0, 0.1)]

counts_00 = []
noncounts_00 = []

for val in cut_values:
    condition = f"{optimized_variable} < {val}"

    counts_00.append(df_CS_tag["00"].Filter(condition).Count())
    noncounts_00.append(df_CS_tag["non00"].Filter(condition).Count())

In [ ]:
all_counts = counts_00 + noncounts_00
ROOT.RDF.RunGraphs(all_counts)

In [ ]:
counts_00 = [int(val.GetValue() * scaling_factor) for val in counts_00]
noncounts_00 = [int(val.GetValue() * scaling_factor) for val in noncounts_00]

In [ ]:
purity = []
significance = []
efficiency = {"00": [], "non00": []}

for non00, neu in zip(noncounts_00, counts_00):
    purity.append(neu / (non00 + neu) if (non00 + neu) > 0 else 0.99)
    significance.append(neu / np.sqrt(non00 + neu) if (non00 + neu) > 0 else 0.0)
    efficiency["00"].append(neu / total_counts["00"]) 
    efficiency["non00"].append(non00 / total_counts["non00"]) 

In [ ]:
# canva.SetRightMargin(0.3)
canva = ROOT.TCanvas("canva", "", 800, 800)

x_arr = np.array(cut_values, dtype=float)
y_arr = np.array(purity, dtype=float)

eff00_arr = np.array(efficiency["00"], dtype=float)
effnon00_arr = np.array(efficiency["non00"], dtype=float)

graph = ROOT.TGraph(len(x_arr), x_arr, y_arr)
graph_00 = ROOT.TGraph(len(x_arr), x_arr, eff00_arr)
graph_non00 = ROOT.TGraph(len(x_arr), x_arr, effnon00_arr)
graph.SetMaximum(1.2)
graph.SetMinimum(0.0)

graph.SetTitle("")
graph.GetXaxis().SetTitle("Cut value #chi^{2}_{tri} [-]")
graph.GetYaxis().SetTitle("Value [-]")
graph.Draw("AL")

graph_00.SetLineColor(ROOT.kRed)
graph_00.Draw("LSAME")

graph_non00.SetLineColor(ROOT.kBlue)
graph_non00.Draw("LSAME")

line = ROOT.TLine(40.0, 0.0, 40.0, 1.2)
line.SetLineStyle(2)
line.Draw("SAME")

legend = ROOT.TLegend(0.5, 0.2, 0.85, 0.45)
legend.AddEntry(graph, "Purity K#rightarrow#pi^{0}#pi^{0}", "l")
legend.AddEntry(graph_00, "Efficiency K#rightarrow#pi^{0}#pi^{0}", "l")
legend.AddEntry(graph_non00, "Efficiency non K#rightarrow#pi^{0}#pi^{0}", "l")
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw()
canva.Draw()

### Checks of counts, purity and efficiency in the optimal threshold

In [ ]:
cut_cond = "true"
optimized_variable = "Chi2TriKinFit"
var_title = "#chi^{2}_{tri} [-]"

In [ ]:
basic_hist = {}
modes = {"full": [201, 0, 200], "zoom": [201, 0, 80]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["00", "non00", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: 00, Index 1: non00, Index 2: data
    
    # Skalowanie histogramów MC (00 i non00)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (00 + non00) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [ ]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 2000, 1000)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow#pi^{0}#pi^{0}", 
    "Events without primary K#rightarrow#pi^{0}#pi^{0}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []
lines = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    line_left = ROOT.TLine(-130.0, 0.0, -130.0, hs.GetMaximum() * 0.5)
    line_right = ROOT.TLine(40.0, 0.0, 40.0, hs.GetMaximum() * 0.5)

    line_right.SetLineStyle(ROOT.kDashed)
    line_left.SetLineStyle(ROOT.kDashed)
    
    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    line_right.Draw("SAME")
    # line_left.Draw("SAME")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)
    lines.append(line_right)
    lines.append(line_left)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

In [ ]:
optimal = 40.0

optimal_purity = graph.Eval(optimal)
optimal_efficiency = graph_00.Eval(optimal)
non00_eff = graph_non00.Eval(optimal)

print(f"Optimal purity: {optimal_purity*100:.2f}%")
print(f"Optimal efficiency: {optimal_efficiency*100:.2f}%")
print(f"non00 efficiency: {non00_eff*100:.2f}%")

In [ ]:
counts_00_arr = np.array(counts_00, dtype=float)
noncounts_00_arr = np.array(noncounts_00, dtype=float)

graph_counts_00 = ROOT.TGraph(len(x_arr), x_arr, counts_00_arr)
graph_noncounts_00 = ROOT.TGraph(len(x_arr), x_arr, noncounts_00_arr)

# ---

optimal_counts_00 = graph_counts_00.Eval(2.0)
optimal_counts_non00 = graph_noncounts_00.Eval(2.0)

print(f"Optimal counts 00: {optimal_counts_00:.0f}")
print(f"Optimal counts non 00: {optimal_counts_non00:.0f}")

In [ ]:
tag_condition += " && Chi2TriKinFit < 40.0"

df_CS_tag["00"] = df_CS_tag["00"].Filter(tag_condition)
df_CS_tag["non00"] = df_CS_tag["non00"].Filter(tag_condition)
df_CS_tag["data"] = df_CS_tag["data"].Filter(tag_condition)

total_counts_tagging = {}

total_counts_tagging["00"] = df_CS_tag["00"].Count()
total_counts_tagging["non00"] = df_CS_tag["non00"].Count()
total_counts_tagging["data"] = df_CS_tag["data"].Count()

ROOT.RDF.RunGraphs(list(total_counts_tagging.values()))

total_counts_tagging = {key: int(val.GetValue() * scaling_factor) if key != 'data' else int(val.GetValue()) for key, val in total_counts_tagging.items()}

In [ ]:
print(total_counts_tagging["data"])

In [ ]:
br_kl_Semileptonic = 0.6759
err_br_kl_Semileptonic = 0.13

print("Counts before tagging:")
total_MC = total_counts['00'] + total_counts['non00']
data_MC_diff = (total_MC - total_counts['data']) / total_counts['data']
purity = total_counts['00'] / (total_counts['00'] + total_counts['non00'])


print(f"MC 00: {total_counts['00']}")
print(f"MC non00: {total_counts['non00']}")
print(f"MC total: {total_MC}")
print(f"Data: {total_counts['data']}")
print(f"(MC - Data) / Data: {100 * data_MC_diff:.2f}%")
print(f"MC 00 purity: {100 * purity:.2f}%")

print("\n")

print("Counts after fiducial volume:")
total_MC = total_counts_fv['00'] + total_counts_fv['non00']
data_MC_diff = (total_MC - total_counts_fv['data']) / total_counts_fv['data']
eff_00 = total_counts_fv['00']/ total_counts['00']
eff_non00 = total_counts_fv['non00']/ total_counts['non00']
eff_MC_total = total_MC / (total_counts['00'] + total_counts['non00'])
eff_Data_total = total_counts_fv['data'] / total_counts['data']
purity = total_counts_fv['00'] / (total_counts_fv['00'] + total_counts_fv['non00'])

print(f"MC 00: {total_counts_fv['00']} (Eff: {100 * eff_00:.2f}%)")
print(f"MC non00: {total_counts_fv['non00']} (Eff: {100 * eff_non00:.2f}%)")
print(f"MC total: {total_MC} (Eff: {100 * eff_MC_total:.2f}%)")
print(f"Data: {total_counts_fv['data']} (Eff: {100 * eff_Data_total:.2f}%)")
print(f"(MC - Data) / Data: {100 * data_MC_diff:.2f}%")
print(f"MC 00 purity: {100 * purity:.2f}%")

print("\n")

print("Counts after tagging:")
total_MC = total_counts_tagging['00'] + total_counts_tagging['non00']
data_MC_diff = (total_MC - total_counts_tagging['data']) / total_counts_tagging['data']
eff_00 = total_counts_tagging['00']/ total_counts['00']
eff_non00 = total_counts_tagging['non00']/ total_counts['non00']
eff_MC_total = total_MC / (total_counts['00'] + total_counts['non00'])
eff_Data_total = total_counts_tagging['data'] / total_counts['data']
purity = total_counts_tagging['00'] / (total_counts_tagging['00'] + total_counts_tagging['non00'])

print(f"MC 00: {total_counts_tagging['00']} (Eff: {100 * eff_00:.2f}%)")
print(f"MC non00: {total_counts_tagging['non00']} (Eff: {100 * eff_non00:.2f}%)")
print(f"MC total: {total_MC} (Eff: {100 * eff_MC_total:.2f}%)")
print(f"Prod KL -> Semileptonic | KS -> pi+pi- in MC: {int(total_MC * br_kl_Semileptonic)}")
print(f"Data: {total_counts_tagging['data']} (Eff: {100 * eff_Data_total:.2f}%)")
print(f"Prod KL -> Semileptonic | KS -> pi+pi- in Data: {int(total_counts_tagging['data'] * br_kl_Semileptonic)}")
print(f"(MC - Data) / Data: {100 * data_MC_diff:.2f}%")
print(f"MC 00 purity: {100 * purity:.2f}%")

### Create the histograms

In [ ]:
cut_cond = "true"
optimized_variable = "Qmiss"
var_title = "Q_{miss} [MeV]"

In [ ]:
basic_hist = {}
modes = {"full": [501, 0.0, 300], "zoom": [501, 0.0, 100.0]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["00", "non00", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: 00, Index 1: non00, Index 2: data
    
    # Skalowanie histogramów MC (00 i non00)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (00 + non00) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [ ]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc

# Opcjonalnie: przeskalowanie histogramu używanego do obliczeń statystyk
# h_calc_obj.Scale(br_kl_Semileptonic)

mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 1000, 500)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow#pi^{0}#pi^{0}", 
    "Events without primary K#rightarrow#pi^{0}#pi^{0}", 
    "Data", 
    "MC sum"
]

canva._stacks = []
canva._legends = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        
        # Przeskalowanie histogramu o br_kl_Semileptonic
        h.Scale(br_kl_Semileptonic)

        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)
            h.SetMarkerSize(0.8)
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe")
        else:
            # Styl dla MC
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")

    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    
    # 2. Rysujemy Dane oraz dopasowujemy skalę Y
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zwiększamy zapas wysokości skali Y do 1.25x (zamiast 1.05x), aby legenda nie zasłaniała danych
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
        hs.SetMaximum(max_y_val)

    leg.Draw()

    canva._stacks.append(hs)
    canva._legends.append(leg)

canva.Update()
canva.Draw()

## Conditional probability: KL->Semileptonic if KS->pi+pi- (change of lookup to channel level)

In [ ]:
df_tag_channel = {}
total_counts_tagged_channels = {}
total_counts_tagged_channels_00 = {}
total_counts_tagged_channels_non00 = {}

for mctruth, chann in channName.items():
    if chann == 'Data':
        df_tag_channel[chann] = df_data.Filter(tag_condition)
        total_counts_tagged_channels[chann] = df_tag_channel[chann].Count()
        total_counts_tagged_channels_00[chann] = df_tag_channel[chann].Filter("has00").Count()
        total_counts_tagged_channels_non00[chann] = df_tag_channel[chann].Filter("!has00").Count()
    elif chann != 'MC sum':
        df_tag_channel[chann] = df.Filter(f"mctruth == {mctruth} && ({tag_condition})")
        total_counts_tagged_channels[chann] = df_tag_channel[chann].Count()
        total_counts_tagged_channels_00[chann] = df_tag_channel[chann].Filter("has00").Count()
        total_counts_tagged_channels_non00[chann] = df_tag_channel[chann].Filter("!has00").Count()

ROOT.RDF.RunGraphs(list(total_counts_tagged_channels.values()))

total_counts_tagged_channels = {key: int(val.GetValue() * scaling_factor) if key != 'Data' else int(val.GetValue()) for key, val in total_counts_tagged_channels.items()}
total_counts_tagged_channels_00 = {key: int(val.GetValue() * scaling_factor) if key != 'Data' else int(val.GetValue()) for key, val in total_counts_tagged_channels_00.items()}
total_counts_tagged_channels_non00 = {key: int(val.GetValue() * scaling_factor) if key != 'Data' else int(val.GetValue()) for key, val in total_counts_tagged_channels_non00.items()}


In [ ]:
def print_purity(counts_dict, label=""):
    # Obliczamy sumę zliczeń wszystkich kanałów MC (pomijamy 'Data' oraz 'MC sum')
    total_mc = sum(
        count
        for ch, count in counts_dict.items()
        if ch not in ["Data", "MC sum"]
    )

    count_Semileptonic = counts_dict.get("Semileptonic", 0)

    if total_mc > 0:
        purity = (count_Semileptonic / total_mc) * 100
        contamination = 100.0 - purity
        print(f"--- Purity analysis {label} ---")
        print(f"Semileptonic counts: {count_Semileptonic}")
        print(f"Total MC counts: {total_mc}")
        print(f"Purity (Semileptonic / Total MC): {purity:.4f}%")
        print(f"Contamination (Background / Total MC): {contamination:.4f}%")
    else:
        print(f"--- Purity analysis {label} ---")
        print("Total MC counts is 0 - cannot calculate purity.")


# --- 1. Total counts after tagging ---
print("Total counts of channels after tagging:")
for chann in channName.values():
    print(f"Counts {chann}: {total_counts_tagged_channels.get(chann, 0)}")

print_purity(total_counts_tagged_channels, "(Total)")
print("\n" + "=" * 50 + "\n")


# --- 2. Total counts after tagging (00) ---
print("Total counts of channels after tagging (00):")
for chann in channName.values():
    print(f"Counts {chann}: {total_counts_tagged_channels_00.get(chann, 0)}")

print_purity(total_counts_tagged_channels_00, "(00)")
print("\n" + "=" * 50 + "\n")


# --- 3. Total counts after tagging (non 00) ---
print("Total counts of channels after tagging (non 00):")
for chann in channName.values():
    print(f"Counts {chann}: {total_counts_tagged_channels_non00.get(chann, 0)}")

print_purity(total_counts_tagged_channels_non00, "(non 00)")

In [ ]:
variables = ["Chi2PMKinFit", "Kchrec_5", "Qmiss"]
var_title = ["#chi^{2}_{fit, +-} [-]", 
             "m^{inv}_{+-} [MeV/c^{2}]",
             "Q_{miss} [MeV]"]

In [ ]:
import ROOT

# Flaga sterująca: True -> odejmij tło !has00 od Danych i bierz tylko has00 dla MC
SUBTRACT_NON_00 = False

basic_hist = {}
modes = {
    "full": [
        [501, 0, 100],
        [501, 280, 600],
        [501, 0, 300]
    ],
    "zoom": [
        [50, 0, 15],
        [251, 420, 530],
        [251, 0, 100]
    ],
}

# Słownik na pomocnicze wskaźniki histogramów !has00 dla Danych (używany gdy SUBTRACT_NON_00 = True)
data_non_00_ptrs = {}

for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}

    for mode, pars in modes.items():

        hist_dict = {}
        for chann in channName.values():
            if chann == "MC sum":
                continue

            h_title = f"hist_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"

            # Przypisujemy pomocniczy dataframe dla danego kanału
            df_aux = df_tag_channel[chann]

            if chann == "Data":
                # Dla Danych tworzymy standardowy histogram
                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )

                # Jeśli odejmujemy !has00, tworzymy dodatkowy histogram filtracyjny dla Data
                if SUBTRACT_NON_00:
                    h_title_non_00 = f"hist_{var}_{mode}_{chann}_non_00"
                    df_non_00_aux = df_aux.Filter("!has00")
                    data_non_00_ptrs[(var, mode)] = df_non_00_aux.Histo1D(
                        (h_title_non_00, hist_title_list, *pars[idx]), var
                    )
            else:
                # Dla MC: jeśli włączone odejmowanie, bierzemy tylko zdarzenia z has00
                if SUBTRACT_NON_00:
                    df_aux = df_aux.Filter("has00")

                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )

        mode_dict[mode] = hist_dict

    basic_hist[var] = mode_dict

# 2. Ewaluacja wszystkich grafów na raz (Lazy Evaluation)
all_ptrs = [
    h_ptr
    for mode_d in basic_hist.values()
    for ch_d in mode_d.values()
    for h_ptr in ch_d.values()
]
if SUBTRACT_NON_00:
    all_ptrs.extend(data_non_00_ptrs.values())

ROOT.RDF.RunGraphs(all_ptrs)

# 3. Odejmowanie tła !has00 od Danych, skalowanie MC i tworzenie "MC sum"
for var, mode_d in basic_hist.items():
    for mode, ch_d in mode_d.items():

        # Odejmujemy histogram !has00 od histogramu Danych
        if SUBTRACT_NON_00 and "Data" in ch_d:
            h_data = ch_d["Data"].GetPtr()
            h_data_non_00 = data_non_00_ptrs[(var, mode)].GetPtr()
            h_data.Add(h_data_non_00, -1.0)

        # Skalowanie kanałów MC (wszystko poza 'Data')
        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        # Tworzenie sumy MC
        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]

        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)

            # Zapisujemy wygenerowaną sumę w słowniku
            ch_d["MC sum"] = h_mc_sum

In [ ]:
canvases = {}

# Pętla po wierszach (każda zmienna 'var' dostaje własny osobny TCanvas)
for var, x_title in zip(variables, var_title):
    
    n_cols = len(basic_hist[var])
    
    # Tworzymy osobne płótno dla danej zmiennej
    c_name = f"canva_{var}"
    c_title = f"Histograms - {var}"
    canva = ROOT.TCanvas(c_name, c_title, 1000 * n_cols, 1000)
    canva.Divide(n_cols, 1)
    
    # Listy do podtrzymania referencji obiektów w pamięci dla tego konkretnego canvasa
    canva._stacks = []
    canva._legends = []

    # Pętla po kolumnach (tryby 'full', 'zoom' itp. dla wybranej zmiennej)
    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):
        
        pad_idx = col_idx + 1
        canva.cd(pad_idx)
        
        hs = ROOT.THStack(f"hs_{var}_{mode_name}", f"{mode_name.capitalize()} mode;{x_title};Counts")
        leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        
        data_hist = None

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            color = channColor.get(chann_key, ROOT.kBlack)
            title = channTitle.get(chann_key, chann_key)
            
            if chann_key == "Data":
                h.SetMarkerStyle(20)
                h.SetMarkerSize(0.8)
                h.SetMarkerColor(color)
                h.SetLineColor(color)
                h.SetLineWidth(2)
                
                data_hist = h
                leg.AddEntry(h, title, "pe")
            else:
                h.SetLineColor(color)
                h.SetLineWidth(2)
                
                hs.Add(h)
                leg.AddEntry(h, title, "l")

        # 1. Rysowanie THStack
        hs.Draw("nostack hist")
        
        # 2. Rysowanie danych i ustawienie właściwej skali Y (1.25x max, żeby nie ucinało piku)
        if data_hist:
            data_hist.Draw("E1 P SAME")
            max_y_val = max(hs.GetMaximum("nostack"), data_hist.GetMaximum()) * 1.25
            hs.SetMaximum(max_y_val)

        leg.Draw()

        # Przypisanie referencji do instancji tego konkretnego Canvasa
        canva._stacks.append(hs)
        canva._legends.append(leg)

    canva.Update()
    canva.Draw()
    
    # Zapis w globalnym słowniku - PyROOT NIE usunie tego z pamięci
    canvases[var] = canva

## Application of cuts

In [ ]:
mctruth_signal = 5

df_tag_Semileptonic = df.Filter(f"mctruth == {mctruth_signal} && {tag_condition}")
df_tag_non_Semileptonic = df.Filter(f"mctruth != {mctruth_signal} && {tag_condition}")
df_tag_data = df_data.Filter(f"{tag_condition}")

In [ ]:
# 2. Add your cuts, including any that use u and v
cuts = [
    "Chi2PMKinFit < 10 * 1.69",
    "abs(Kchrec_5 - 497.605) < 12 * 0.837"
]

cut_counts = []
cut_counts_bkg = []
cut_eff = []
cut_Semileptonic_eff = []
rel_cut_eff = []
rel_cut_Semileptonic_eff = []

basis_Semileptonic_counts = 0

for chann, val in total_counts_tagged_channels.items():
    if chann == 'Data' or chann == 'MC sum':
        continue
        
    basis_Semileptonic_counts += val

basis_Semileptonic_mctruth_counts = total_counts_tagged_channels["Semileptonic"]
basis_non_Semileptonic_mctruth_counts = basis_Semileptonic_counts - total_counts_tagged_channels["Semileptonic"]

basis_Semileptonic_counts *= br_kl_Semileptonic
basis_Semileptonic_counts = int(basis_Semileptonic_counts)

purity = []

combined_cut = ""

for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
    cut_counts.append(df_tag_Semileptonic.Filter(combined_cut).Count())
    cut_counts_bkg.append(df_tag_non_Semileptonic.Filter(combined_cut).Count())

tot_cuts = cut_counts + cut_counts_bkg
ROOT.RDF.RunGraphs(tot_cuts)

cut_counts = np.array([int(c.GetValue() * scaling_factor) for c in cut_counts])
cut_counts_bkg = np.array([int(c.GetValue() * scaling_factor) for c in cut_counts_bkg])

tot_cuts = cut_counts + cut_counts_bkg

print(f"--- Beginning MC cuts ---")
print(f"Total prod Semileptonic events: {basis_Semileptonic_counts}")
print(f"Total prod Semileptonic mctruth events: {basis_Semileptonic_mctruth_counts}")
print(f"Total prod non Semileptonic mctruth events: {basis_non_Semileptonic_mctruth_counts}")
print(f"\n--- Start of cuts ---")


for i, count in enumerate(cut_counts):
    eff = tot_cuts[i] / basis_Semileptonic_counts if basis_Semileptonic_counts > 0 else 1.0
    rel_eff = tot_cuts[i] / tot_cuts[i-1] if i > 0 and tot_cuts[i-1] > 0 else eff
    eff_Semileptonic = cut_counts[i] / basis_Semileptonic_mctruth_counts if basis_Semileptonic_mctruth_counts > 0 else 1.0
    rel_eff_Semileptonic = cut_counts[i] / cut_counts[i-1] if i > 0 and cut_counts[i-1] > 0 else eff_Semileptonic
    pur = (cut_counts[i]) / (cut_counts[i] + cut_counts_bkg[i]) if (cut_counts[i] + cut_counts_bkg[i]) != 0 else 1.0
    cut_eff.append(eff)
    rel_cut_eff.append(rel_eff)
    cut_Semileptonic_eff.append(eff_Semileptonic)
    rel_cut_Semileptonic_eff.append(rel_eff_Semileptonic)
    purity.append(pur)

    message = f'''Cut {cuts[i]}:
                  Total Semileptonic registered events: {tot_cuts[i]} ({cut_eff[i]*100:.3f}%) <--> ({rel_cut_eff[i]*100:.3f}%)
                  Semileptonic from MC Truth: {cut_counts[i]} ({cut_Semileptonic_eff[i]*100:.3f}%) <--> ({rel_cut_Semileptonic_eff[i]*100:.3f}%)
                  Non Semileptonic from MC Truth: {cut_counts_bkg[i]}
                  Purity: {purity[i]*100:.3f}%
    '''

    print(message)


In [ ]:
# # 2. Add your cuts, including any that use u and v

cut_counts = []
cut_eff = []
rel_cut_eff = []

combined_cut = ""

basis_data_counts = total_counts_tagged_channels["Data"] * br_kl_Semileptonic

for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
    cut_counts.append(df_tag_data.Filter(combined_cut).Count())

ROOT.RDF.RunGraphs(cut_counts)

for i, count in enumerate(cut_counts):
    cut_counts[i] = count.GetValue()
    eff = cut_counts[i] / basis_data_counts if basis_data_counts > 0 else 1.0
    rel_eff = cut_counts[i] / cut_counts[i-1] if i > 0 and cut_counts[i-1] > 0 else eff
    cut_eff.append(eff)
    rel_cut_eff.append(rel_eff)

    message = f'''Cut {cuts[i]}:
                  Total Semileptonic registered events: {cut_counts[i]} ({cut_eff[i]*100:.3f}%) <--> ({rel_cut_eff[i]*100:.3f}%)
    '''

    print(message)

In [ ]:
cuts = [
    "Chi2PMKinFit < 10 * 1.69",
    "abs(Kchrec_5 - 497.605) < 12.0 * 0.837"
]

In [ ]:
variables = ["Chi2PMKinFit", "Kchrec_5", "Qmiss"]
var_title = ["#chi^{2}_{fit, +-00} [-]", 
             "m^{inv}_{+-} [MeV/c^{2}]",
             "Q_{miss} [MeV]"]

In [ ]:
import ROOT

# Flaga sterująca: True -> odejmij tło MC z !has00 od Danych i zachowaj tylko MC z has00
SUBTRACT_NON_00 = False

combined_cut = ""

for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"

print(combined_cut)

basic_hist = {}
modes = {
    "full": [
        [501, 0, 1000],
        [501, 280, 600],
        [501, 0, 300]
    ],
    "zoom": [
        [50, 0, 100],
        [251, 480, 510],
        [251, 0, 100]
    ],
}

# Słownik na pomocnicze wskaźniki histogramów MC !has00 (do odjęcia od Danych)
mc_non_00_ptrs = {}

for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}

    for mode, pars in modes.items():

        hist_dict = {}
        non_00_list = []

        for chann in channName.values():
            if chann == "MC sum":
                continue

            h_title = f"hist_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"

            # Pomocnicza zmienna RDataFrame
            df_aux = df_tag_channel[chann].Filter(combined_cut)

            if chann == "Data":
                # Dla Danych po prostu pobieramy histogram
                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )
            else:
                if SUBTRACT_NON_00:
                    # Tło MC z !has00 rejestrujemy osobno do odjęcia od Danych
                    h_title_non_00 = f"hist_{var}_{mode}_{chann}_non_00"
                    df_non_00_aux = df_aux.Filter("!has00")
                    non_00_list.append(
                        df_non_00_aux.Histo1D(
                            (h_title_non_00, hist_title_list, *pars[idx]), var
                        )
                    )

                    # Do głównego słownika trafia tylko MC z has00
                    df_aux = df_aux.Filter("has00")

                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )

        if SUBTRACT_NON_00:
            mc_non_00_ptrs[(var, mode)] = non_00_list

        mode_dict[mode] = hist_dict

    basic_hist[var] = mode_dict

# 2. Ewaluacja wszystkich grafów na raz (Lazy Evaluation)
all_ptrs = [
    h_ptr
    for mode_d in basic_hist.values()
    for ch_d in mode_d.values()
    for h_ptr in ch_d.values()
]
if SUBTRACT_NON_00:
    all_ptrs.extend(
        [h_ptr for ptr_list in mc_non_00_ptrs.values() for h_ptr in ptr_list]
    )

ROOT.RDF.RunGraphs(all_ptrs)

# 3. Odejmowanie tła MC (!has00) od Danych, skalowanie MC i tworzenie "MC sum"
for var, mode_d in basic_hist.items():
    for mode, ch_d in mode_d.items():

        # Odejmujemy przeskalowane tło MC (!has00) od histogramu Danych
        if SUBTRACT_NON_00 and "Data" in ch_d:
            h_data = ch_d["Data"].GetPtr()
            for h_non_00_ptr in mc_non_00_ptrs[(var, mode)]:
                h_data.Add(
                    h_non_00_ptr.GetPtr(), -1.0 * scaling_factor
                )  # Odjęcie z uwzględnieniem skalowania MC

        # Skalowanie głównych kanałów MC (wszystko poza 'Data')
        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        # Tworzenie sumy MC
        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]

        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)

            # Zapisujemy wygenerowaną sumę w słowniku
            ch_d["MC sum"] = h_mc_sum

In [ ]:
canvases = []

# Pętla po zmiennych – każda zmienna tworzy nowy Canvas
for row_idx, (var, x_title) in enumerate(zip(variables, var_title)):

    # Pobranie n_cols bez ryzyka błędu i rzutowanie szerokości na int
    n_cols = len(basic_hist[var])
    width = int(500 * n_cols)
    height = 400

    canva = ROOT.TCanvas(f"canva_{var}", f"Canvas {var}", width, height)
    canva.Divide(n_cols, 1)

    # Przypisujemy listy obiektów bezpośrednio do instancji płótna (zapobiega usuwaniu przez Python GC i segfaultom)
    canva._stacks = []
    canva._legends = []

    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):

        canva.cd(col_idx + 1)

        hs = ROOT.THStack(
            f"hs_{var}_{mode_name}",
            f"{mode_name.capitalize()} mode;{x_title};Counts",
        )
        leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)

        data_hist = None

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            color = channColor.get(chann_key, ROOT.kBlack)
            title = channTitle.get(chann_key, chann_key)

            if chann_key == "Data":
                # Styl dla danych eksperymentalnych (PE1)
                h.SetMarkerStyle(20)
                h.SetMarkerSize(0.8)
                h.SetMarkerColor(color)
                h.SetLineColor(color)
                h.SetLineWidth(2)

                data_hist = h
                leg.AddEntry(h, title, "pe")
            else:
                # Styl dla MC
                h.SetLineColor(color)
                h.SetLineWidth(2)

                hs.Add(h)
                leg.AddEntry(h, title, "l")

        # 1. Rysujemy stack z MC
        hs.Draw("nostack hist")

        # 2. Rysujemy Dane oraz dopasowujemy skalę Y
        if data_hist:
            data_hist.Draw("E1 P SAME")
            # Zwiększamy zapas wysokości do 1.25x, aby legenda nie zasłaniała wykresu (mnożnik 0.6 spłaszczał wykres)
            max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
            hs.SetMaximum(max_y_val)

        leg.Draw()

        # Zachowanie referencji per canvas
        canva._stacks.append(hs)
        canva._legends.append(leg)

    canva.Update()
    canva.Draw()
    canvases.append(canva)

In [ ]:
import ROOT

# Wyłączenie automatycznego powiązywania obiektów z otwartym plikiem ROOT
ROOT.TH1.AddDirectory(False)
ROOT.gErrorIgnoreLevel = ROOT.kWarning
ROOT.gStyle.SetOptStat(0)

# ==========================================
# KONFIGURACJA I PARAMETRY
# ==========================================
purity_threshold = 0.75      # Domyślny próg czystości (używany jeśli brak wpisu manualnego)
max_rel_error = 0.20         # Maksymalny dopuszczalny błąd względny stosunku wydajności (10%)
br_kl_Semileptonic = 1.0 #0.6759         # Branching ratio KL -> Semileptonic
output_root_filename = "fits_results.root"

SAVE_CONFIG = {
    "save_fits": True,        # Zapisuj wyekstrapolowane funkcje TF1
    "save_ratios": True,      # Zapisuj histogramy stosunku wydajności
    "save_efficiencies": True,# Zapisuj histogramy wydajności Data/MC
    "save_purity": True       # Zapisuj histogramy czystości
}

# RĘCZNE ZAKRESY FITOWANIA: (var, mode) -> (x_min, x_max)
MANUAL_FIT_RANGES = {
    ("Qmiss", "full"): (20.0, 40.0),
    ("Qmiss", "zoom"): (20.0, 40.0),
}

# Słowniki pomocnicze na histogramy mianowników (po samym tagowaniu)
tag_hist = {}
mc_tag_non_00_ptrs = {}

# 1. TWORZENIE HISTOGRAMÓW MIANOWNIKÓW
for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}
    for mode, pars in modes.items():
        hist_dict = {}
        non_00_list = []

        for chann in channName.values():
            if chann == "MC sum":
                continue

            h_title = f"hist_tag_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"
            df_aux = df_tag_channel[chann]

            if chann == "Data":
                hist_dict[chann] = df_aux.Histo1D((h_title, hist_title_list, *pars[idx]), var)
            else:
                if SUBTRACT_NON_00:
                    df_non_00_aux = df_aux.Filter("!has00")
                    non_00_list.append(df_non_00_aux.Histo1D((f"{h_title}_non_00", hist_title_list, *pars[idx]), var))
                    df_aux = df_aux.Filter("has00")

                hist_dict[chann] = df_aux.Histo1D((h_title, hist_title_list, *pars[idx]), var)

        if SUBTRACT_NON_00:
            mc_tag_non_00_ptrs[(var, mode)] = non_00_list

        mode_dict[mode] = hist_dict
    tag_hist[var] = mode_dict

# 2. LAZY EVALUATION MIANOWNIKÓW
tag_ptrs = [h_ptr for m_d in tag_hist.values() for c_d in m_d.values() for h_ptr in c_d.values()]
if SUBTRACT_NON_00:
    tag_ptrs.extend([h_ptr for ptr_l in mc_tag_non_00_ptrs.values() for h_ptr in ptr_l])

ROOT.RDF.RunGraphs(tag_ptrs)

# 3. KOREKTA I SKALOWANIE MIANOWNIKÓW
for var, mode_d in tag_hist.items():
    for mode, ch_d in mode_d.items():
        if SUBTRACT_NON_00 and "Data" in ch_d:
            h_data = ch_d["Data"].GetPtr()
            for h_non_00_ptr in mc_tag_non_00_ptrs[(var, mode)]:
                h_data.Add(h_non_00_ptr.GetPtr(), -1.0 * scaling_factor)

        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]
        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_tag_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)
            ch_d["MC sum"] = h_mc_sum


# ==========================================
# OBROBKA WYDAJNOSCI, STOSUNKU I FITOWANIE
# ==========================================
canvases_eff = []
canvases_ratio = []
canvases_purity = []
fitted_functions = {}
persistent_objects = []

ROOT.gDirectory.cd("/")
root_file = ROOT.TFile(output_root_filename, "RECREATE")

print(f"{'Zmienna':<20} | {'Mode':<10} | {'p0 (Intercept)':<20} | {'p1 (Slope)':<20} | {'Chi2/NDF':<10}")
print("-" * 90)

for var, x_title in zip(variables, var_title):
    n_cols = len(basic_hist[var])
    
    ROOT.gDirectory.cd("/")
    c_eff = ROOT.TCanvas(f"canva_eff_{var}", f"Efficiencies - {var}", 550 * n_cols, 420)
    c_eff.Divide(n_cols, 1)
    
    c_ratio = ROOT.TCanvas(f"canva_ratio_eff_{var}", f"Data/MC Eff Ratio Fit - {var}", 550 * n_cols, 420)
    c_ratio.Divide(n_cols, 1)

    c_purity = ROOT.TCanvas(f"canva_purity_{var}", f"Channel Purity - {var}", 550 * n_cols, 420)
    c_purity.Divide(n_cols, 1)

    c_eff._graphics = []
    c_ratio._graphics = []
    c_purity._graphics = []

    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):
        
        # Pobieranie histogramów liczników i mianowników
        h_data_num = chann_dict["Data"].GetPtr() if hasattr(chann_dict["Data"], "GetPtr") else chann_dict["Data"]
        h_mc_num = chann_dict["MC sum"].GetPtr() if hasattr(chann_dict["MC sum"], "GetPtr") else chann_dict["MC sum"]
        h_Semileptonic_num = chann_dict["Semileptonic"].GetPtr() if hasattr(chann_dict["Semileptonic"], "GetPtr") else chann_dict["Semileptonic"]

        h_data_denom_raw = tag_hist[var][mode_name]["Data"].GetPtr() if hasattr(tag_hist[var][mode_name]["Data"], "GetPtr") else tag_hist[var][mode_name]["Data"]
        h_mc_denom_raw = tag_hist[var][mode_name]["MC sum"].GetPtr() if hasattr(tag_hist[var][mode_name]["MC sum"], "GetPtr") else tag_hist[var][mode_name]["MC sum"]

        # Skalowanie mianowników o branching ratio
        h_data_denom = h_data_denom_raw.Clone(f"h_data_denom_{var}_{mode_name}")
        h_data_denom.Scale(br_kl_Semileptonic)

        h_mc_denom = h_mc_denom_raw.Clone(f"h_mc_denom_{var}_{mode_name}")
        h_mc_denom.Scale(br_kl_Semileptonic)

        # Wydajności
        h_eff_data = h_data_num.Clone(f"h_eff_data_{var}_{mode_name}")
        h_eff_data.Divide(h_data_num, h_data_denom, 1.0, 1.0, "B")

        h_eff_mc = h_mc_num.Clone(f"h_eff_mc_{var}_{mode_name}")
        h_eff_mc.Divide(h_mc_num, h_mc_denom, 1.0, 1.0, "B")

        # Czystość po cięciach
        h_purity = h_Semileptonic_num.Clone(f"h_purity_{var}_{mode_name}")
        h_purity.Reset()
        h_purity.SetTitle(f"Purity #pi^{{#pm}}l^{{#mp}}#nu / MC Sum ({mode_name});{x_title};Purity")
        h_purity.Divide(h_Semileptonic_num, h_mc_num, 1.0, 1.0, "B")

        # Stosunek wydajności i zakreślanie wg Czystości + Błędu
        h_eff_ratio = h_eff_data.Clone(f"h_eff_ratio_{var}_{mode_name}")
        h_eff_ratio.Reset()
        h_eff_ratio.SetTitle(f"Ratio Eff_Data / Eff_MC ({mode_name});{x_title};Eff Data / Eff MC")

        auto_x_min = float('inf')
        auto_x_max = float('-inf')

        for ibin in range(1, h_eff_ratio.GetNbinsX() + 1):
            eff_d = h_eff_data.GetBinContent(ibin)
            eff_m = h_eff_mc.GetBinContent(ibin)
            err_d = h_eff_data.GetBinError(ibin)
            err_m = h_eff_mc.GetBinError(ibin)

            if eff_m > 0 and eff_d > 0:
                val_r = eff_d / eff_m
                err_r = val_r * ROOT.TMath.Sqrt((err_d / eff_d)**2 + (err_m / eff_m)**2)
                h_eff_ratio.SetBinContent(ibin, val_r)
                h_eff_ratio.SetBinError(ibin, err_r)

                # Względny błąd binowania: err_r / val_r
                rel_err = err_r / val_r
                purity_val = h_purity.GetBinContent(ibin)

                # WARUNEK PODWÓJNY: Czystość ORAZ Kontrola Fluktuacji (Błąd Względny)
                if purity_val >= purity_threshold and rel_err <= max_rel_error:
                    bin_center = h_eff_ratio.GetBinCenter(ibin)
                    auto_x_min = min(auto_x_min, bin_center)
                    auto_x_max = max(auto_x_max, bin_center)

        # Wybór zakresu
        manual_range = MANUAL_FIT_RANGES.get((var, mode_name), None)
        if manual_range is not None:
            fit_x_min, fit_x_max = manual_range
        else:
            fit_x_min, fit_x_max = auto_x_min, auto_x_max

        # -------------------------------------------------------------
        # RYSOWANIE CANVAS 1: WYDAJNOŚCI
        # -------------------------------------------------------------
        pad_e = c_eff.cd(col_idx + 1)
        pad_e.SetLeftMargin(0.14)
        pad_e.SetBottomMargin(0.13)
        pad_e.SetRightMargin(0.05)

        h_eff_data.SetTitle(f"Efficiencies ({mode_name});{x_title};Efficiency")
        h_eff_data.GetYaxis().SetTitleOffset(1.3)
        h_eff_data.SetMarkerStyle(20)
        h_eff_data.SetMarkerSize(0.8)
        h_eff_data.SetMarkerColor(ROOT.kBlack)
        h_eff_data.SetLineColor(ROOT.kBlack)
        h_eff_data.SetMinimum(0.0)
        h_eff_data.SetMaximum(1.2)
        h_eff_data.Draw("E1 P")

        purity_box = None
        if fit_x_min < fit_x_max:
            purity_box = ROOT.TBox(fit_x_min, 0.0, fit_x_max, 1.2)
            purity_box.SetFillColorAlpha(ROOT.kGreen + 1, 0.20)
            purity_box.Draw("SAME")

            l_min = ROOT.TLine(fit_x_min, 0.0, fit_x_min, 1.2)
            l_max = ROOT.TLine(fit_x_max, 0.0, fit_x_max, 1.2)
            for line_b in (l_min, l_max):
                line_b.SetLineColor(ROOT.kGreen + 2)
                line_b.SetLineStyle(2)
                line_b.SetLineWidth(2)
                line_b.Draw("SAME")
            c_eff._graphics.extend([purity_box, l_min, l_max])

        h_eff_data.Draw("E1 P SAME")

        h_eff_mc.SetLineColor(ROOT.kRed)
        h_eff_mc.SetLineWidth(2)
        h_eff_mc.Draw("HIST SAME")

        ROOT.gPad.RedrawAxis()

        leg_eff = ROOT.TLegend(0.42, 0.68, 0.85, 0.88)
        leg_eff.SetBorderSize(0)
        leg_eff.SetFillStyle(0)
        leg_eff.SetTextSize(0.035)
        leg_eff.AddEntry(h_eff_data, "Data Eff", "pe")
        leg_eff.AddEntry(h_eff_mc, "MC Sum Eff", "l")
        if purity_box:
            leg_eff.AddEntry(purity_box, "Fit Range Region", "f")
        leg_eff.Draw()

        c_eff._graphics.extend([leg_eff, h_eff_data, h_eff_mc])

        # -------------------------------------------------------------
        # RYSOWANIE CANVAS 2: STOSUNEK I FIT LINIOWY
        # -------------------------------------------------------------
        pad_r = c_ratio.cd(col_idx + 1)
        pad_r.SetLeftMargin(0.14)
        pad_r.SetBottomMargin(0.13)
        pad_r.SetRightMargin(0.05)

        h_eff_ratio.GetYaxis().SetTitleOffset(1.3)
        h_eff_ratio.SetMarkerStyle(20)
        h_eff_ratio.SetMarkerSize(0.8)
        h_eff_ratio.SetMinimum(0.5)
        h_eff_ratio.SetMaximum(1.5)
        h_eff_ratio.Draw("E1 P")

        # Zielony obszar dopasowania na wykresie Stosunku (Ratio)
        if fit_x_min < fit_x_max:
            ratio_box = ROOT.TBox(fit_x_min, 0.5, fit_x_max, 1.5)
            ratio_box.SetFillColorAlpha(ROOT.kGreen + 1, 0.20)
            ratio_box.Draw("SAME")

            l_ratio_min = ROOT.TLine(fit_x_min, 0.5, fit_x_min, 1.5)
            l_ratio_max = ROOT.TLine(fit_x_max, 0.5, fit_x_max, 1.5)
            for line_r in (l_ratio_min, l_ratio_max):
                line_r.SetLineColor(ROOT.kGreen + 2)
                line_r.SetLineStyle(2)
                line_r.SetLineWidth(2)
                line_r.Draw("SAME")

            c_ratio._graphics.extend([ratio_box, l_ratio_min, l_ratio_max])

            # Przerysowanie punktów ratio, aby nie były zasłonięte przez TBox
            h_eff_ratio.Draw("E1 P SAME")

            # Fitowanie funkcji
            fit_name = f"fit_eff_ratio_{var}_{mode_name}"
            f_fit = ROOT.TF1(fit_name, "pol0", fit_x_min, fit_x_max)
            f_fit.SetLineColor(ROOT.kBlue)
            f_fit.SetLineWidth(2)
            fit_res = h_eff_ratio.Fit(f_fit, "R Q S")

            x_min_full = h_eff_ratio.GetXaxis().GetXmin()
            x_max_full = h_eff_ratio.GetXaxis().GetXmax()
            
            f_extrap = ROOT.TF1(f"extrap_{fit_name}", "pol0", x_min_full, x_max_full)
            f_extrap.SetParameters(f_fit.GetParameters())
            f_extrap.SetLineColor(ROOT.kRed)
            f_extrap.SetLineStyle(2)
            f_extrap.SetLineWidth(2)
            
            f_extrap.Draw("SAME")
            f_fit.Draw("SAME")

            fitted_functions[(var, mode_name)] = f_extrap

            p0, p0_err = f_fit.GetParameter(0), f_fit.GetParError(0)
            p1, p1_err = f_fit.GetParameter(1), f_fit.GetParError(1)
            chi2_ndf = f_fit.GetChisquare() / max(1, f_fit.GetNDF())
            
            print(f"{var:<20} | {mode_name:<10} | {p0:.4f} +/- {p0_err:.4f} | {p1:.6f} +/- {p1_err:.6f} | {chi2_ndf:.2f}")

            root_file.cd()
            if SAVE_CONFIG["save_fits"]:
                f_extrap.Write(f"fit_{var}_{mode_name}")
            if SAVE_CONFIG["save_ratios"]:
                h_eff_ratio.Write(f"ratio_{var}_{mode_name}")
            if SAVE_CONFIG["save_efficiencies"]:
                h_eff_data.Write(f"eff_data_{var}_{mode_name}")
                h_eff_mc.Write(f"eff_mc_{var}_{mode_name}")
            if SAVE_CONFIG["save_purity"]:
                h_purity.Write(f"purity_{var}_{mode_name}")

            c_ratio._graphics.extend([f_fit, f_extrap])

        line_ratio = ROOT.TLine(h_eff_ratio.GetXaxis().GetXmin(), 1.0, h_eff_ratio.GetXaxis().GetXmax(), 1.0)
        line_ratio.SetLineStyle(3)
        line_ratio.Draw("SAME")
        c_ratio._graphics.append(line_ratio)

        # -------------------------------------------------------------
        # RYSOWANIE CANVAS 3: HISTOGRAM CZYSTOŚCI
        # -------------------------------------------------------------
        pad_p = c_purity.cd(col_idx + 1)
        pad_p.SetLeftMargin(0.14)
        pad_p.SetBottomMargin(0.13)
        pad_p.SetRightMargin(0.05)

        h_purity.GetYaxis().SetTitleOffset(1.3)
        h_purity.SetMarkerStyle(21)
        h_purity.SetMarkerSize(0.8)
        h_purity.SetMarkerColor(ROOT.kBlue + 2)
        h_purity.SetLineColor(ROOT.kBlue + 2)
        h_purity.SetMinimum(0.0)
        h_purity.SetMaximum(1.1)
        h_purity.Draw("E1 P")

        line_purity_thresh = ROOT.TLine(h_purity.GetXaxis().GetXmin(), purity_threshold, h_purity.GetXaxis().GetXmax(), purity_threshold)
        line_purity_thresh.SetLineColor(ROOT.kRed)
        line_purity_thresh.SetLineStyle(7)
        line_purity_thresh.Draw("SAME")

        if fit_x_min < fit_x_max:
            purity_fit_box = ROOT.TBox(fit_x_min, 0.0, fit_x_max, 1.1)
            purity_fit_box.SetFillColorAlpha(ROOT.kGreen + 1, 0.15)
            purity_fit_box.Draw("SAME")
            h_purity.Draw("E1 P SAME")
            c_purity._graphics.append(purity_fit_box)

        c_purity._graphics.extend([line_purity_thresh, h_purity])

        persistent_objects.extend([h_eff_data, h_eff_mc, h_eff_ratio, h_purity])

    c_eff.Update()
    c_eff.Draw()
    canvases_eff.append(c_eff)

    c_ratio.Update()
    c_ratio.Draw()
    canvases_ratio.append(c_ratio)

    c_purity.Update()
    c_purity.Draw()
    canvases_purity.append(c_purity)

root_file.Close()

print("-" * 90)
print(f"Pomyślnie zapisano plik: {output_root_filename}")

In [ ]:
import ROOT

# Wyłączenie automatycznego powiązywania obiektów z otwartym plikiem ROOT
ROOT.TH1.AddDirectory(False)
ROOT.gErrorIgnoreLevel = ROOT.kWarning

# ==========================================
# KONFIGURACJA I PARAMETRY
# ==========================================
purity_threshold = 0.75      # Domyślny próg czystości (używany jeśli brak wpisu manualnego)
max_rel_error = 0.20         # Maksymalny dopuszczalny błąd względny stosunku wydajności (10%)
br_kl_Semileptonic = 1.0 #0.6759         # Branching ratio KL -> Semileptonic        # Branching ratio KL -> Semileptonic
output_root_filename = "fits_results.root"

SAVE_CONFIG = {
    "save_fits": True,        # Zapisuj wyekstrapolowane funkcja TF1
    "save_ratios": True,      # Zapisuj histogramy stosunku wydajności
    "save_efficiencies": True,# Zapisuj histogramy wydajności Data/MC
    "save_purity": True       # Zapisuj histogramy czystości
}

# RĘCZNE ZAKRESY FITOWANIA: (var, mode) -> (x_min, x_max)
MANUAL_FIT_RANGES = {
    ("Qmiss", "full"): (20.0, 40.0),
    ("Qmiss", "zoom"): (20.0, 40.0),
}

# Słowniki pomocnicze na histogramy mianowników (po samym tagowaniu)
tag_hist = {}
mc_tag_non_pm_ptrs = {}

# 1. TWORZENIE HISTOGRAMÓW MIANOWNIKÓW
for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}
    for mode, pars in modes.items():
        hist_dict = {}
        non_pm_list = []

        for chann in channName.values():
            if chann == "MC sum":
                continue

            h_title = f"hist_tag_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"
            df_aux = df_tag_channel[chann]

            if chann == "Data":
                hist_dict[chann] = df_aux.Histo1D((h_title, hist_title_list, *pars[idx]), var)
            else:
                if SUBTRACT_NON_00:
                    df_non_pm_aux = df_aux.Filter("!hasPM")
                    non_pm_list.append(df_non_pm_aux.Histo1D((f"{h_title}_non_pm", hist_title_list, *pars[idx]), var))
                    df_aux = df_aux.Filter("hasPM")

                hist_dict[chann] = df_aux.Histo1D((h_title, hist_title_list, *pars[idx]), var)

        if SUBTRACT_NON_00:
            mc_tag_non_pm_ptrs[(var, mode)] = non_pm_list

        mode_dict[mode] = hist_dict
    tag_hist[var] = mode_dict

# 2. LAZY EVALUATION MIANOWNIKÓW
tag_ptrs = [h_ptr for m_d in tag_hist.values() for c_d in m_d.values() for h_ptr in c_d.values()]
if SUBTRACT_NON_00:
    tag_ptrs.extend([h_ptr for ptr_l in mc_tag_non_pm_ptrs.values() for h_ptr in ptr_l])

ROOT.RDF.RunGraphs(tag_ptrs)

# 3. KOREKTA I SKALOWANIE MIANOWNIKÓW
for var, mode_d in tag_hist.items():
    for mode, ch_d in mode_d.items():
        if SUBTRACT_NON_00 and "Data" in ch_d:
            h_data = ch_d["Data"].GetPtr()
            for h_non_pm_ptr in mc_tag_non_pm_ptrs[(var, mode)]:
                h_data.Add(h_non_pm_ptr.GetPtr(), -1.0 * scaling_factor)

        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]
        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_tag_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)
            ch_d["MC sum"] = h_mc_sum


# ==========================================
# OBROBKA WYDAJNOSCI, STOSUNKU I FITOWANIE
# ==========================================
canvases_eff = []
canvases_ratio = []
canvases_purity = []
fitted_functions = {}
persistent_objects = []

ROOT.gDirectory.cd("/")
root_file = ROOT.TFile(output_root_filename, "RECREATE")

print(f"{'Zmienna':<20} | {'Mode':<10} | {'p0 (Intercept)':<20} | {'p1 (Slope)':<20} | {'Chi2/NDF':<10}")
print("-" * 90)

for var, x_title in zip(variables, var_title):
    n_cols = len(basic_hist[var])
    
    ROOT.gDirectory.cd("/")
    c_eff = ROOT.TCanvas(f"canva_eff_{var}", f"Efficiencies - {var}", 550 * n_cols, 420)
    c_eff.Divide(n_cols, 1)
    
    c_ratio = ROOT.TCanvas(f"canva_ratio_eff_{var}", f"Data/MC Eff Ratio Fit - {var}", 550 * n_cols, 420)
    c_ratio.Divide(n_cols, 1)

    c_purity = ROOT.TCanvas(f"canva_purity_{var}", f"Channel Purity - {var}", 550 * n_cols, 420)
    c_purity.Divide(n_cols, 1)

    c_eff._graphics = []
    c_ratio._graphics = []
    c_purity._graphics = []

    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):
        
        # Pobieranie histogramów liczników
        h_data_num = chann_dict["Data"].GetPtr() if hasattr(chann_dict["Data"], "GetPtr") else chann_dict["Data"]
        h_mc_num = chann_dict["MC sum"].GetPtr() if hasattr(chann_dict["MC sum"], "GetPtr") else chann_dict["MC sum"]
        h_Semileptonic_num = chann_dict["Semileptonic"].GetPtr() if hasattr(chann_dict["Semileptonic"], "GetPtr") else chann_dict["Semileptonic"]

        # Pobieranie raw histogramów mianowników (po tagowaniu)
        h_data_denom_raw = tag_hist[var][mode_name]["Data"].GetPtr() if hasattr(tag_hist[var][mode_name]["Data"], "GetPtr") else tag_hist[var][mode_name]["Data"]
        h_Semileptonic_denom_raw = tag_hist[var][mode_name]["Semileptonic"].GetPtr() if hasattr(tag_hist[var][mode_name]["Semileptonic"], "GetPtr") else tag_hist[var][mode_name]["Semileptonic"]

        # -------------------------------------------------------------
        # 1. OBLICZANIE MIANOWNIKÓW I WYDAJNOŚCI (JEDEN RAZ)
        # -------------------------------------------------------------
        presel_eff_mc = 1.0  # Wydajność preselekcji/tagowania w MC

        # DANE:
        h_data_denom = h_data_denom_raw.Clone(f"h_data_denom_{var}_{mode_name}")
        h_data_denom.Scale(br_kl_Semileptonic)

        h_eff_data = h_data_num.Clone(f"h_eff_data_{var}_{mode_name}")
        h_eff_data.Divide(h_data_num, h_data_denom, 1.0, 1.0, "B")

        # MC:
        h_mc_denom = h_Semileptonic_denom_raw.Clone(f"h_mc_denom_{var}_{mode_name}")
        h_mc_denom.Scale(1.0 / presel_eff_mc)

        h_Semileptonic_num_clean = h_Semileptonic_num.Clone(f"h_Semileptonic_num_clean_{var}_{mode_name}")

        h_eff_mc = h_Semileptonic_num_clean.Clone(f"h_eff_mc_{var}_{mode_name}")
        h_eff_mc.Divide(h_Semileptonic_num_clean, h_mc_denom, 1.0, 1.0, "B")

        # CZYSTOŚĆ:
        h_purity = h_Semileptonic_num.Clone(f"h_purity_{var}_{mode_name}")
        h_purity.Reset()
        h_purity.SetTitle(f"Purity #pi^{{#pm}}l^{{#mp}}#nu^{{0}} / MC Sum ({mode_name});{x_title};Purity")
        h_purity.Divide(h_Semileptonic_num, h_mc_num, 1.0, 1.0, "B")

        # STOSUNEK (RATIO):
        h_eff_ratio = h_eff_data.Clone(f"h_eff_ratio_{var}_{mode_name}")
        h_eff_ratio.Reset()
        h_eff_ratio.SetTitle(f"Ratio Eff_Data / Eff_MC ({mode_name});{x_title};Eff Data / Eff MC")

        auto_x_min = float('inf')
        auto_x_max = float('-inf')

        for ibin in range(1, h_eff_ratio.GetNbinsX() + 1):
            eff_d = h_eff_data.GetBinContent(ibin)
            eff_m = h_eff_mc.GetBinContent(ibin)
            err_d = h_eff_data.GetBinError(ibin)
            err_m = h_eff_mc.GetBinError(ibin)

            if eff_m > 0 and eff_d > 0:
                val_r = eff_d / eff_m
                err_r = val_r * ROOT.TMath.Sqrt((err_d / eff_d)**2 + (err_m / eff_m)**2)
                h_eff_ratio.SetBinContent(ibin, val_r)
                h_eff_ratio.SetBinError(ibin, err_r)

                rel_err = err_r / val_r
                purity_val = h_purity.GetBinContent(ibin)

                if purity_val >= purity_threshold and rel_err <= max_rel_error:
                    bin_center = h_eff_ratio.GetBinCenter(ibin)
                    auto_x_min = min(auto_x_min, bin_center)
                    auto_x_max = max(auto_x_max, bin_center)

        manual_range = MANUAL_FIT_RANGES.get((var, mode_name), None)
        if manual_range is not None:
            fit_x_min, fit_x_max = manual_range
        else:
            fit_x_min, fit_x_max = auto_x_min, auto_x_max

        # -------------------------------------------------------------
        # RYSOWANIE CANVAS 1: WYDAJNOŚCI (DODANE!)
        # -------------------------------------------------------------
        pad_e = c_eff.cd(col_idx + 1)
        pad_e.SetLeftMargin(0.14)
        pad_e.SetBottomMargin(0.13)
        pad_e.SetRightMargin(0.05)

        h_eff_data.SetTitle(f"Efficiencies ({mode_name});{x_title};Efficiency")
        h_eff_data.GetYaxis().SetTitleOffset(1.3)
        h_eff_data.SetMarkerStyle(20)
        h_eff_data.SetMarkerSize(0.8)
        h_eff_data.SetMarkerColor(ROOT.kBlack)
        h_eff_data.SetLineColor(ROOT.kBlack)
        h_eff_data.SetMinimum(0.0)
        h_eff_data.SetMaximum(1.2)
        h_eff_data.Draw("E1 P")

        purity_box = None
        if fit_x_min < fit_x_max:
            lower_limit = max(fit_x_min, h_eff_data.GetXaxis().GetXmin())
            upper_limit = min(fit_x_max, h_eff_data.GetXaxis().GetXmax())
            
            purity_box = ROOT.TBox(lower_limit, 0.0, upper_limit, 1.2)
            purity_box.SetFillColorAlpha(ROOT.kGreen + 1, 0.20)
            purity_box.Draw("SAME")

            l_min = ROOT.TLine(lower_limit, 0.0, lower_limit, 1.2)
            l_max = ROOT.TLine(upper_limit, 0.0, upper_limit, 1.2)
            for line_b in (l_min, l_max):
                line_b.SetLineColor(ROOT.kGreen + 2)
                line_b.SetLineStyle(2)
                line_b.SetLineWidth(2)
                line_b.Draw("SAME")
            c_eff._graphics.extend([purity_box, l_min, l_max])

        h_eff_data.Draw("E1 P SAME")

        h_eff_mc.SetLineColor(ROOT.kRed)
        h_eff_mc.SetLineWidth(2)
        h_eff_mc.Draw("HIST SAME")

        leg_eff = ROOT.TLegend(0.42, 0.68, 0.85, 0.88)
        leg_eff.SetBorderSize(0)
        leg_eff.SetFillStyle(0)
        leg_eff.SetTextSize(0.035)
        leg_eff.AddEntry(h_eff_data, "Data Eff", "pe")
        leg_eff.AddEntry(h_eff_mc, "MC #pi^{{#pm}}l^{{#mp}}#nu^{0} Eff", "l")
        if purity_box:
            leg_eff.AddEntry(purity_box, "Fit Range Region", "f")
        leg_eff.Draw()

        c_eff._graphics.extend([leg_eff, h_eff_data, h_eff_mc])

        # -------------------------------------------------------------
        # RYSOWANIE CANVAS 2: STOSUNEK I FIT LINIOWY
        # -------------------------------------------------------------
        pad_r = c_ratio.cd(col_idx + 1)
        pad_r.SetLeftMargin(0.14)
        pad_r.SetBottomMargin(0.13)
        pad_r.SetRightMargin(0.05)

        h_eff_ratio.GetYaxis().SetTitleOffset(1.3)
        h_eff_ratio.SetMarkerStyle(20)
        h_eff_ratio.SetMarkerSize(0.8)
        h_eff_ratio.SetMinimum(0.5)
        h_eff_ratio.SetMaximum(1.5)
        h_eff_ratio.Draw("E1 P")

        if fit_x_min < fit_x_max:
            lower_limit = max(fit_x_min, h_eff_ratio.GetXaxis().GetXmin())
            upper_limit = min(fit_x_max, h_eff_ratio.GetXaxis().GetXmax())
            
            ratio_box = ROOT.TBox(lower_limit, 0.5, upper_limit, 1.5)
            ratio_box.SetFillColorAlpha(ROOT.kGreen + 1, 0.20)
            ratio_box.Draw("SAME")

            l_ratio_min = ROOT.TLine(lower_limit, 0.5, lower_limit, 1.5)
            l_ratio_max = ROOT.TLine(upper_limit, 0.5, upper_limit, 1.5)
            for line_r in (l_ratio_min, l_ratio_max):
                line_r.SetLineColor(ROOT.kGreen + 2)
                line_r.SetLineStyle(2)
                line_r.SetLineWidth(2)
                line_r.Draw("SAME")

            c_ratio._graphics.extend([ratio_box, l_ratio_min, l_ratio_max])
            h_eff_ratio.Draw("E1 P SAME")

            fit_name = f"fit_eff_ratio_{var}_{mode_name}"
            f_fit = ROOT.TF1(fit_name, "pol0", fit_x_min, fit_x_max)
            f_fit.SetLineColor(ROOT.kBlue)
            f_fit.SetLineWidth(2)
            fit_res = h_eff_ratio.Fit(f_fit, "R Q S")

            x_min_full = h_eff_ratio.GetXaxis().GetXmin()
            x_max_full = h_eff_ratio.GetXaxis().GetXmax()
            
            f_extrap = ROOT.TF1(f"extrap_{fit_name}", "pol0", x_min_full, x_max_full)
            f_extrap.SetParameters(f_fit.GetParameters())
            f_extrap.SetLineColor(ROOT.kRed)
            f_extrap.SetLineStyle(2)
            f_extrap.SetLineWidth(2)
            
            f_extrap.Draw("SAME")
            f_fit.Draw("SAME")

            fitted_functions[(var, mode_name)] = f_extrap

            p0, p0_err = f_fit.GetParameter(0), f_fit.GetParError(0)
            p1, p1_err = f_fit.GetParameter(1), f_fit.GetParError(1)
            chi2_ndf = f_fit.GetChisquare() / max(1, f_fit.GetNDF())
            
            print(f"{var:<20} | {mode_name:<10} | {p0:.4f} +/- {p0_err:.4f} | {p1:.6f} +/- {p1_err:.6f} | {chi2_ndf:.2f}")

            root_file.cd()
            if SAVE_CONFIG["save_fits"]:
                f_extrap.Write(f"fit_{var}_{mode_name}")
            if SAVE_CONFIG["save_ratios"]:
                h_eff_ratio.Write(f"ratio_{var}_{mode_name}")
            if SAVE_CONFIG["save_efficiencies"]:
                h_eff_data.Write(f"eff_data_{var}_{mode_name}")
                h_eff_mc.Write(f"eff_mc_{var}_{mode_name}")
            if SAVE_CONFIG["save_purity"]:
                h_purity.Write(f"purity_{var}_{mode_name}")

            c_ratio._graphics.extend([f_fit, f_extrap])

        line_ratio = ROOT.TLine(h_eff_ratio.GetXaxis().GetXmin(), 1.0, h_eff_ratio.GetXaxis().GetXmax(), 1.0)
        line_ratio.SetLineStyle(3)
        line_ratio.Draw("SAME")
        c_ratio._graphics.append(line_ratio)

        # -------------------------------------------------------------
        # RYSOWANIE CANVAS 3: HISTOGRAM CZYSTOŚCI
        # -------------------------------------------------------------
        pad_p = c_purity.cd(col_idx + 1)
        pad_p.SetLeftMargin(0.14)
        pad_p.SetBottomMargin(0.13)
        pad_p.SetRightMargin(0.05)

        h_purity.GetYaxis().SetTitleOffset(1.3)
        h_purity.SetMarkerStyle(21)
        h_purity.SetMarkerSize(0.8)
        h_purity.SetMarkerColor(ROOT.kBlue + 2)
        h_purity.SetLineColor(ROOT.kBlue + 2)
        h_purity.SetMinimum(0.0)
        h_purity.SetMaximum(1.1)
        h_purity.Draw("E1 P")

        line_purity_thresh = ROOT.TLine(h_purity.GetXaxis().GetXmin(), purity_threshold, h_purity.GetXaxis().GetXmax(), purity_threshold)
        line_purity_thresh.SetLineColor(ROOT.kRed)
        line_purity_thresh.SetLineStyle(7)
        line_purity_thresh.Draw("SAME")

        if fit_x_min < fit_x_max:
            lower_limit = max(fit_x_min, h_purity.GetXaxis().GetXmin())
            upper_limit = min(fit_x_max, h_purity.GetXaxis().GetXmax())
            
            purity_fit_box = ROOT.TBox(lower_limit, 0.0, upper_limit, 1.1)
            purity_fit_box.SetFillColorAlpha(ROOT.kGreen + 1, 0.15)
            purity_fit_box.Draw("SAME")
            h_purity.Draw("E1 P SAME")
            c_purity._graphics.append(purity_fit_box)

        c_purity._graphics.extend([line_purity_thresh, h_purity])

        persistent_objects.extend([h_eff_data, h_eff_mc, h_eff_ratio, h_purity])

    c_eff.Update()
    c_eff.Draw()
    canvases_eff.append(c_eff)

    c_ratio.Update()
    c_ratio.Draw()
    canvases_ratio.append(c_ratio)

    c_purity.Update()
    c_purity.Draw()
    canvases_purity.append(c_purity)

root_file.Close()

print("-" * 90)
print(f"Pomyślnie zapisano plik: {output_root_filename}")